# B2.12 · Building the DAST and exploitation harness

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

Builds on **[B2.11 · Building the SAST harness](https://spbreed.github.io/cyber-commons/lessons/B2.11.html)**.

| | |
|---|---|
| Open-source tooling | OWASP ZAP, Metasploit, CAI |
| Open-weight models | GLM-5.2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Static analysis produces a hypothesis. Dynamic analysis produces a fact — but
only if the thing deciding is not the model.

The DAST harness is the same loop with a different last step: drive a running
target, generate a payload, observe what happened, and confirm on a
**deterministic oracle**. Not "the model thinks this worked". A signal that
exists independently of anything the agent believes:

- the query the database actually executed
- the process that actually spawned
- the file that actually opened
- the sleep that actually delayed the response by five seconds

The distinction is not pedantic. A model asked "did that work?" after sending a
payload will answer confidently either way, and its confidence is uncorrelated
with the truth. An oracle that reads the target's own logs cannot be talked
into anything.

Three things bound this phase, and they are preconditions rather than
afterthoughts: **sandbox replication** (a replica, never production), a
**safe-action policy** (demonstrate, never damage), and **blast-radius limits**
(the exploit stops at proof). Evidence capture is what makes it a finding
instead of an anecdote.

## 2 · A target that records what actually happened\n\nThe oracle is the target's own log, not the agent's opinion of it.

In [ ]:
SANDBOX = {"is_replica": True, "has_prod_credentials": False,
           "network_egress": False}

class Target:
    """A stand-in application. Its log is the oracle."""
    def __init__(self):
        self.log = []
    def handle(self, path, param):
        if path == "/report":
            q = f"SELECT * FROM reports WHERE name = '{param}'"
            self.log.append(("query", q))
            return {"status": 200, "body": "report data"}
        if path == "/export":
            cmd = f"exporter --format {param}"
            self.log.append(("exec", cmd))
            return {"status": 200, "body": "ok"}
        return {"status": 404, "body": ""}

def preconditions_ok(sb):
    fails = [k for k, v in (("is_replica", sb["is_replica"]),
                            ("no_prod_credentials", not sb["has_prod_credentials"]),
                            ("no_egress", not sb["network_egress"])) if not v]
    return (not fails), fails

ok, fails = preconditions_ok(SANDBOX)
print(f"sandbox preconditions: {'PASS' if ok else 'REFUSE ' + str(fails)}")
assert ok, "the harness must refuse to run outside a sandbox"

## 3 · Drive it, and confirm on the oracle

In [ ]:
PAYLOADS = {
 "/report": ["normal", "x' OR '1'='1", "'; DROP TABLE reports; --"],
 "/export": ["pdf", "pdf; id", "pdf && whoami"],
}

def oracle_sqli(entry):    return entry[0] == "query" and ("OR '1'='1" in entry[1] or "DROP" in entry[1])
def oracle_cmdi(entry):    return entry[0] == "exec" and (";" in entry[1] or "&&" in entry[1])
ORACLES = {"/report": ("CWE-89", oracle_sqli), "/export": ("CWE-78", oracle_cmdi)}

confirmed = []
for path, payloads in sorted(PAYLOADS.items()):
    cwe, oracle = ORACLES[path]
    for p in payloads:
        t = Target()
        resp = t.handle(path, p)
        fired = any(oracle(e) for e in t.log)
        mark = "CONFIRMED" if fired else "no        "
        print(f"   {path:9s}{p[:26]:28s}status {resp['status']}  {mark}")
        if fired:
            confirmed.append({"path": path, "cwe": cwe, "payload": p,
                              "observable": t.log[-1][1]})
print(f"\nconfirmed by oracle: {len(confirmed)}")

## 4 · Where it breaks — the model as its own oracle\n\nSame payloads, same target. The only change is who decides.

In [ ]:
def model_opinion(path, payload, response):
    """NOT a language model - a stub reproducing the failure mode: it reads the
    response, sees HTTP 200, and calls that success."""
    return response["status"] == 200

model_confirmed = []
for path, payloads in sorted(PAYLOADS.items()):
    for p in payloads:
        t = Target()
        resp = t.handle(path, p)
        if model_opinion(path, p, resp):
            model_confirmed.append((path, p))

print(f"oracle confirmed : {len(confirmed)}")
print(f"model confirmed  : {len(model_confirmed)}")
print()
false_positives = [(p, pl) for p, pl in model_confirmed
                   if not any(c["path"] == p and c["payload"] == pl for c in confirmed)]
print(f"findings the model asserted that the oracle refuses: {len(false_positives)}")
for p, pl in false_positives:
    print(f"   {p:9s}{pl}")
print()
print("Every one of these returns HTTP 200 because the application works. The")
print("model is not lying; it was asked a question it has no way to answer, and")
print("it answered anyway.")
assert len(model_confirmed) > len(confirmed)

## 5 · The control — stop at proof, and capture the evidence

In [ ]:
SAFE_ACTIONS = {"read", "observe", "time"}
DESTRUCTIVE = {"drop", "delete", "truncate", "shutdown"}

def safe_to_send(payload):
    low = payload.lower()
    hit = sorted(d for d in DESTRUCTIVE if d in low)
    return (not hit), hit

print(f"{'payload':32s}{'sendable?':11s}why")
for path, payloads in sorted(PAYLOADS.items()):
    for p in payloads:
        ok, hit = safe_to_send(p)
        print(f"{p[:30]:32s}{'yes' if ok else 'REFUSED':11s}"
              f"{'demonstrates without damage' if ok else 'destructive: ' + str(hit)}")
print()
print("The DROP TABLE payload proves nothing that OR '1'='1' has not already")
print("proved, and it destroys the replica you need for the next test.")
print("Demonstrating the flaw and exercising it are different jobs.")
assert not safe_to_send("'; DROP TABLE reports; --")[0]

## 6 · Verify — an evidence record somebody can re-run

In [ ]:
evidence = [{
   "cwe": c["cwe"], "entry": c["path"], "input": c["payload"],
   "observable": c["observable"],                    # the oracle's own words
   "oracle": "target execution log",
   "sandbox": {"replica": True, "prod_credentials": False, "egress": False},
   "reproducible": True,
} for c in confirmed if safe_to_send(c["payload"])[0]]

for e in evidence:
    print(f"   {e['cwe']}  {e['entry']}")
    print(f"      input      : {e['input']}")
    print(f"      observable : {e['observable']}")
print()
print(f"{len(evidence)} findings, each with the observable that proves it.")
print("An exit code is not an observable. The query the database ran is.")
assert evidence and all(e["observable"] for e in evidence)

## What you just proved

Six payloads are driven at a sandboxed replica. The deterministic oracle — the target's own execution log — confirms the injections and refuses the benign requests, while a stub standing in for model self-assessment confirms every request that returned HTTP 200, including the ones that prove nothing. The destructive payload is refused before it is sent, and each surviving finding carries the observable that proves it.

## Your turn

Take one 'confirmed' finding from a tool you run and ask what the oracle was. If the answer is the model's own summary of the response, you have a hypothesis with a confident tone.

---

**Next → [B2.13 · Building the threat-modelling harness](https://spbreed.github.io/cyber-commons/lessons/B2.13.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.12.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.12.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*